In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
torch.set_num_threads(5)

/home/pwiesenbach/anaconda3/envs/BertGCN/lib/python3.6/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import datetime
import logging
import pickle
import random
from pathlib import Path

import dgl
import torch
import torch.utils.data as Data
from ignite.engine import Engine, Events
from ignite.handlers import EarlyStopping, ModelCheckpoint
from ignite.handlers.param_scheduler import LRScheduler, create_lr_scheduler_with_warmup
from ignite.metrics import Accuracy, ClassificationReport, Loss
from ignite.utils import setup_logger
from torch.optim.lr_scheduler import ExponentialLR, ReduceLROnPlateau
from transformers import AutoTokenizer

from clinic_datasets import CleanClinicDataset
from metrics import SklearnClassificationReport
from model import BertGAT, BertGCN
from params import parse_args
from utils import *

Using backend: pytorch


In [22]:
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)

MODELPATH = "deepset/gbert-base"
BERTLR = 5e-5
GCNLR = 3e-5
LR = 4e-5
BATCHSIZE = 8
NEPOCHS = 50
ACCUSTEPS = 8
LOGINTERVALL = 100
DATASET = "med_indication_all_RF_diag"
SAVEPATH = "models/gcn/gbert-base_med_indication_all_RF_diag_best.pt"

tokenizer = AutoTokenizer.from_pretrained(MODELPATH)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

logging.basicConfig(
    format=f"%(asctime)s - %(message)s",
    # format=f"%(asctime)s ({args.mixfactor}) - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[
        logging.StreamHandler(),
    ],
)

In [18]:
dataset_file = Path("data") / "medindcls_bert.json"
if not dataset_file.exists():
    logging.info("Creating dataset")
    dataset = CleanClinicDataset(tokenizer=tokenizer, clean=False)
    with open(dataset_file, "wb") as f:
        logging.info(f"Saving dataset under {dataset_file}")
        pickle.dump(dataset, f)
else:
    logging.info(f"Loading dataset from: {dataset_file}")
    with open(dataset_file, "rb") as f:
        dataset = pickle.load(f)
        
idx = np.arange(len(dataset))
random.shuffle(idx)
train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
    )

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(
    DATASET
)

nb_node = features.shape[0]
nb_train, nb_val, nb_test = train_mask.sum(), val_mask.sum(), test_mask.sum()
nb_word = nb_node - nb_train - nb_val - nb_test
nb_class = y_train.shape[1]


model = BertGCN(
    nb_class=nb_class,
    pretrained_model="deepset/gbert-base",
    #mix_factor=trial.suggest_uniform("mix_factor", 0, 1),
    mix_factor=0.5,
    gcn_layers=2,
    n_hidden=200,
    dropout=0.5,
)

model.load_state_dict(torch.load(SAVEPATH, map_location=torch.device('cpu')))

# transform one-hot label to class ID for pytorch computation
y = y_train + y_test + y_val
y_train = y_train.argmax(axis=1)
y = y.argmax(axis=1)

# document mask used for update feature
doc_mask = train_mask + val_mask + test_mask

tokenizer = AutoTokenizer.from_pretrained(MODELPATH)

input_ids = torch.cat(
    [
        torch.tensor(np.array([x["input_ids"] for x in np.array(dataset.examples)[train_idx]])),
        torch.zeros((nb_word, tokenizer.model_max_length), dtype=torch.long),
        torch.tensor(np.array([x["input_ids"] for x in np.array(dataset.examples)[val_idx]])),
        torch.tensor(np.array([x["input_ids"] for x in np.array(dataset.examples)[test_idx]])),
    ]
)

adj_norm = normalize_adj(adj + sp.eye(adj.shape[0]))
    
test_idx_dataset = Data.TensorDataset(torch.arange(nb_node - nb_test, nb_node, dtype=torch.long))
    
idx_loader_test = Data.DataLoader(test_idx_dataset, batch_size=BATCHSIZE)

criterion = torch.nn.CrossEntropyLoss()

graph = dgl.from_scipy(adj_norm.astype("float32"), eweight_name="edge_weight")
graph.ndata["input_ids"] = input_ids
graph.ndata["label"], graph.ndata["train"], graph.ndata["val"], graph.ndata["test"] = (
    torch.LongTensor(y),
    torch.FloatTensor(train_mask),
    torch.FloatTensor(val_mask),
    torch.FloatTensor(test_mask),
)
graph.ndata["label_train"] = torch.LongTensor(y_train)
graph.ndata["cls_feats"] = torch.zeros((nb_node, model.feat_dim))

2022-10-26 14:33:19 - Loading dataset from: data/medindcls_bert.json


(1889, 768) (1889, 12) (270, 768) (270, 12) (540, 768) (540, 12) (27186, 768) (27186, 12)
27996


In [21]:
def update_feature():
    global graph, model
    dataloader = Data.DataLoader(Data.TensorDataset(graph.ndata["input_ids"][doc_mask]), batch_size=64)
    with torch.no_grad():
        model = model.to(device)
        model.eval()
        cls_list = []
        logging.info("Udating features...")
        for batch in dataloader:
            input_ids = [x.to(device) for x in batch][0]
            output = model.bert_model(input_ids=input_ids)[0][:, 0]
            cls_list.append(output.cpu())
        cls_feat = torch.cat(cls_list, axis=0)
    graph = graph.to("cpu")
    graph.ndata["cls_feats"][doc_mask] = cls_feat

def eval_step(engine, batch):
    global model, graph
    with torch.no_grad():
        model.eval()
        model = model.to(device)
        graph = graph.to(device)
        (idx,) = [x.to(device) for x in batch]
        y_pred = model(graph, idx)
        y_true = graph.ndata["label"][idx]
        return y_pred, y_true

trainer = Engine(eval_step)

metrics = {
    "accuracy": Accuracy(),
    "nll": Loss(criterion),
    "cr": SklearnClassificationReport(target_names=dataset.LE.classes_)
}

for n, f in metrics.items():
    f.attach(trainer, n)

@trainer.on(Events.COMPLETED)
def log_test_results(trainer):
    metrics = trainer.state.metrics
    logging.info(
        f"Test Results - Epoch[{trainer.state.epoch}] Avg accuracy: {metrics['accuracy']:.2f} Avg loss: {metrics['nll']:.2f}"
    )
    logging.info(metrics["cr"])

update_feature()
trainer.run(idx_loader_test)

2022-10-26 15:01:33 - Udating features...
2022-10-26 15:16:40 - Engine run starting with max_epochs=1.
2022-10-26 15:17:55 - Epoch[1] Complete. Time taken: 00:01:15.132
2022-10-26 15:17:55 - Test Results - Epoch[1] Avg accuracy: 0.22 Avg loss: 4.64
2022-10-26 15:17:55 -                                     precision    recall  f1-score   support

         Blutdrucksenker_Blutdruck       0.33      0.35      0.34       171
          Blutdrucksenker_Herzschw       0.09      0.08      0.08        39
  Blutdrucksenker_Hypercholesterin       0.28      0.26      0.27       163
            Blutdrucksenker_beides       0.00      0.00      0.00        20
            Blutdrucksenker_unklar       0.00      0.00      0.00         8
Cholesterinsenker_Hypercholesterin       0.07      0.09      0.08        22
             Cholesterinsenker_KHK       0.17      0.20      0.19        64
          Cholesterinsenker_beides       0.00      0.00      0.00         7
          Cholesterinsenker_unklar       0.0

State:
	iteration: 68
	epoch: 1
	epoch_length: 68
	max_epochs: 1
	output: <class 'tuple'>
	batch: <class 'list'>
	metrics: <class 'dict'>
	dataloader: torch.utils.data.dataloader.DataLoader
	seed: <class 'NoneType'>
	times: <class 'dict'>